# 🛡️ Deepfake Detector — Backend (GPU)
Run this notebook on Google Colab with **GPU runtime** to get fast inference.

**Steps:**
1. Runtime → Change runtime type → **T4 GPU**
2. Run all cells (Ctrl+F9)
3. Copy the ngrok URL printed at the end
4. Set it as `VITE_API_URL` in your frontend `.env`

In [1]:
# Cell 1: Clone your repo
!git clone https://github.com/Tanish024/Deep.git
%cd Deep/backend

Cloning into 'Deep'...
remote: Enumerating objects: 68, done.
remote: Counting objects: 100% (68/68), done.
remote: Compressing objects: 100% (55/55), done.
remote: Total 68 (delta 14), reused 65 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (68/68), 70.50 KiB | 17.62 MiB/s, done.
Resolving deltas: 100% (14/14), done.
/content/Deep/backend


In [2]:
# Cell 2: Install dependencies (GPU version of PyTorch auto-installed on Colab)
!pip install fastapi uvicorn transformers huggingface_hub \
    opencv-python-headless python-multipart pydantic Pillow numpy pyngrok

In [3]:
# Cell 3: Test GPU availability
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('⚠️ No GPU! Go to Runtime → Change runtime type → T4 GPU')

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [4]:
# Cell 4: Patch detector.py to use GPU if available
import os
os.environ['HF_HOME'] = './models'
os.environ['FAKE_THRESHOLD'] = '0.5'

# Read and patch detector to use cuda
detector_path = 'app/detector.py'
with open(detector_path, 'r') as f:
    code = f.read()

code = code.replace(
    '_DEVICE = "cpu"',
    '_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"'
)
code = 'import torch\n' + code  # ensure torch is imported before the device line

with open(detector_path, 'w') as f:
    f.write(code)

print('✅ Patched detector.py for GPU')

✅ Patched detector.py for GPU


In [5]:
# Cell 5: Quick test — load model and run inference
from app.detector import classify_frame
from PIL import Image
import numpy as np

img = Image.fromarray(np.zeros((224, 224, 3), dtype=np.uint8))
score = classify_frame(img)
print(f'✅ Model works! Test score: {score:.4f}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.torchscript:   0%|          | 0.00/607M [00:00<?, ?B/s]

Loading TorchScript model from /content/Deep/backend/models/models--yermandy--deepfake-detection/snapshots/9a6857ec642deb57373c5437be803a199468b8c6/model.torchscript ...
Loading CLIPProcessor (openai/clip-vit-large-patch14) ...


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.52k [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/961k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

Model loaded successfully
✅ Model works! Test score: 0.1689


In [6]:
# Cell 6: Setup ngrok tunnel (exposes your Colab backend to the internet)
# Get a free auth token at: https://dashboard.ngrok.com/signup
# Then paste it below:

NGROK_AUTH_TOKEN = "3FJTZwXlLgNIcHc5K340D7JCArd_3WDpopXtG5wJdZGC1PHH3"  # <-- PASTE YOUR TOKEN

from pyngrok import ngrok
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
public_url = ngrok.connect(8000)
print(f'\n🌐 Your backend is live at:')
print(f'   {public_url}')
print(f'\n📋 Copy this URL and set it as VITE_API_URL in your frontend .env')
print(f'   VITE_API_URL={public_url}')


🌐 Your backend is live at:
   NgrokTunnel: "https://recliner-mobile-customer.ngrok-free.dev" -> "http://localhost:8000"

📋 Copy this URL and set it as VITE_API_URL in your frontend .env
   VITE_API_URL=NgrokTunnel: "https://recliner-mobile-customer.ngrok-free.dev" -> "http://localhost:8000"


In [9]:
# Cell 7: Start the FastAPI server (runs forever — keep this cell running!)
# nest_asyncio fixes 'cannot be called from a running event loop' in Colab
!pip install nest_asyncio -q
import nest_asyncio
nest_asyncio.apply()

import uvicorn
uvicorn.run('app.main:app', host='0.0.0.0', port=8000)

/usr/lib/python3.12/pathlib.py:404: RuntimeWarning: coroutine 'Server.serve' was never awaited
  parsed = [sys.intern(str(x)) for x in rel.split(sep) if x and x != '.']


RuntimeError: asyncio.run() cannot be called from a running event loop